# Eurobarometer Data Extraction

As mentioned before, the valuable Eurobarometer data comes in a messy `.xlsx` format, which is not usable for the purposes of data analysis. The idea is to come up with a solution for a messy Excel file with multiple tabs, because the newest Eurobarometer Surveys are only presented in this format. The data requires a lot of transformation.

**The problems**:
- impossible to export a given Excel file to `.csv` because of multiple tabs; the solution is to read the file in a loop, read each tab and write it into separated `.csv` (to merge them in the future);
- each tab should be edited: it has a lot of unnecessary info (like text, questions in French, etc); these have to be removed;
- structure of the tabs is not the same (e.g., there are tabs with 'open questions' or similar);
- column Total (easy to remove in comparison to other transformation);
- rows are duplicated: the info fiven in absolute numbers (number of informants who gave a certain answer) and percentage (of people in a given country who answered the question in a particular way); the rows with absolute numbers should be removed;
- countries are columns but should be rows;
- each indicator (e.g., trust to a national government) have multiple dimensions (fully trust (1), somewhat trust (2), neutral (3), somewhat do not trust (4), absolutely don't trust (5), don't know (6) etc); the amount of answers is not the same for each question; it should be adjusted:
    - option 1: keep everything (if it's easier); than each indicator will be converted to many (Depending on how many alternatives the survey participants were given);
    - option 2: drop those who don't know or neutral and keep only those who are 'positive' (e.g., take those who are fully trust + somewhat trust and make a new indicator);
    - option 3: probably there are other options; but the idea is to get ONE number per indicator per country
- YEAR column: has to be added.

There are several tasks which need to be adressed in this notebook:
- downloading raw data for three Eurobarometers (metadata file with links/names of the tables will be provided);
- transforming `.xlsx` files into machine-readable `.csv` files;
- uploading them to PostgreSQL.

In the current notebook all the examples will be implemented on a `.xlsx` file, representing Special Eurobarometer "Digital Decade" (no. SP566, 2025).

## 1. Reading Tabs | Example

I'm going to work with the EU countries only, but this part could be skipped by those who want to include in the analysis all the data:

In [8]:
eu_countries = [
    'BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 'EL', 'ES', 'FR', 
    'HR', 'IT', 'CY', 'LV', 'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 
    'PL', 'PT', 'RO', 'SI', 'SK', 'FI', 'SE'
]

We can get several tabs from the file are check how they look like / try to read them and transform into a dataframe, which will allow us saving the tab later as a `.csv` file.

I will read the following tabs:
- 'B': List of countries,
- 'QE1_5': Engaging in democratic life via digital technology, importance
- 'QE2': Digitalization of daily life: makes it easier or more difficult
- 'C2': Political Interest Index

In [ ]:
import pandas as pd
import re

# tabs ro read:
specific_tabs = ['B', 'QE1_5', 'QE2', 'C2']

# dictionary of dataframes: 
dict_of_dfs = pd.read_excel('./eurobarometer_data/eb_sp566.xlsx', sheet_name=specific_tabs, header=8)

# reading the tabs separately and printing their names:
for sheet_name, df in dict_of_dfs.items():
    print(f"Reading tab: {sheet_name}")

Reading tab: B
Reading tab: QE1_5
Reading tab: QE2
Reading tab: C2


#### B: Countries
Now we can take a closer look at the `B` (Countries) tab:

In [13]:
df_countries = dict_of_dfs['B']
df_countries.head(5)

,<<Back to content,Unnamed: 1,UE27\nEU27,BE,BG,CZ,DK,DEW,DE,DEE,...,MT,NL,AT,PL,PT,RO,SI,SK,FI,SE
0,NaN,Total,26319,1003,1018,1005,1004,1216,1510,294,...,503,1021,1008,1008,1053,1039,1010,1006,1001,1020
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,AT,538,0,0,0,0,0,0,0,...,0,0,1008,0,0,0,0,0,0,0
3,NaN,AT,0.02,-,-,-,-,-,-,-,...,-,-,1,-,-,-,-,-,-,-
4,NaN,BE,679,1003,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


This dataframe already looks pretty messy:
- columns that we don't need (e.g., "Back to content")
- multiple rows that do not serve any purpouse (all the info is avaliable in row 1)
- etc.

We can get rid of all of it and save a clean dataframe as a `.csv` file (in case we need to know, how many participants from each country answered the Eurobarometer questions).

In [ ]:
# start over:
df_countries = dict_of_dfs['B']

# drop useless columns:
df_countries = df_countries.drop(columns=['<<Back to content', 'UE27\nEU27'])

# grab only one row and make it to pivot the table
df_countries = df_countries.iloc[[0]].T
df_countries = df_countries.iloc[1:]
df_countries = df_countries.reset_index()

# change the column names:
df_countries = df_countries.rename(columns={
    'index': 'country',
    0: 'num_ind'
})

# drop the non-EU countries:
df_countries = df_countries[df_countries['country'].isin(eu_countries)]

# check the result:
df_countries.head(10)
df_countries.shape

# write to a file:
# df_countries.to_csv('B.csv', index=False)

(27, 2)

#### C2: Political Interest Index

In [42]:
df_polit_ind = dict_of_dfs['C2']
df_polit_ind = df_polit_ind.drop(columns=['<<Back to content', 'UE27\nEU27'])
df_polit_ind.rename(columns={'Unnamed: 1':'intensity_c2'},inplace=True)
symbol_mapping = {
    'Total':'Total',
    '+ +':'high',
    '+':'medium',
    '-':'low',
    '- -':'very low'
}
df_polit_ind['intensity_c2']=df_polit_ind['intensity_c2'].replace(symbol_mapping)
num_cols = df_polit_ind.columns.drop('intensity_c2')
df_polit_ind[num_cols] = df_polit_ind[num_cols].apply(pd.to_numeric, errors='coerce')
df_polit_ind = df_polit_ind[df_polit_ind['BE']<1]
df_polit_ind = df_polit_ind.T
df_polit_ind.columns = df_polit_ind.iloc[0]
df_polit_ind = df_polit_ind.drop(df_polit_ind.index[0])
df_polit_ind.reset_index(inplace=True)
df_polit_ind.rename(columns={'index': 'country'}, inplace=True)
df_polit_ind.name = None
df_polit_ind

intensity_c2,country,high,medium,low,very low
0,BE,0.15,0.49,0.2,0.16
1,BG,0.21,0.58,0.13,0.08
2,CZ,0.09,0.56,0.15,0.2
3,DK,0.29,0.46,0.17,0.08
4,DEW,0.24,0.56,0.16,0.04
5,DE,0.25,0.56,0.15,0.04
6,DEE,0.26,0.55,0.15,0.04
7,EE,0.19,0.48,0.18,0.15
8,IE,0.18,0.45,0.2,0.17
9,EL,0.27,0.49,0.12,0.12
